## Las cuatro fuentes del proyecto

El estudio cruza **cuatro conjuntos de datos** para responder qué tan lejos está la compra de vivienda para una persona joven (18-34) en el Reino Unido, desde tres ángulos: cuánto hay que ahorrar (años de salario), cuánto cuesta pagarla a crédito y qué papel juega el clima en los precios.

**Fase 2 · fuentes y tipos de datos (guía).** En este cuaderno simplemente cargamos las cuatro fuentes, describimos su estructura y dejamos fijada la ventana de años en la que las series se pueden confrontar sin inventar periodos.

In [1]:
import sys
from pathlib import Path
_ROOT = Path.cwd()
while not any((_ROOT / c).exists() for c in ['db', 'data']) and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'script'))
from proyecto import graficos
display(graficos.fig_fases())

<Figure size 1344x364 with 1 Axes>

| # | Fuente | Archivo | Qué aporta |
|---|---|---|---|
| F1 | Ventas de vivienda (Land Registry + geografía) | parquet limpio previo | Precio realmente pagado por región y ciudad |
| F2 | Oferta hipotecaria (snapshot 2022, Kaggle) | `data/crudo/UK_Mortgage_Rate.csv` | Tasas, comisiones y plazos de 7.796 productos |
| F3 | Clima Met Office (36 estaciones) | `data/crudo/MET_Office_Weather_Data.csv` | Temperaturas, lluvia, heladas y sol |
| F4 | Precio medio + salario mediana | `Average_UK_houseprices_and_salary.csv` | Serie nacional 1975-2020 ajustada a GBP-2020 |
| F4b | Salario por edad y género | `Income_by_age_and_gender.csv` | Brecha salarial de 2021 entre bandas |

## 1. Buscar la raíz del proyecto

Todo arranca fijando la carpeta raíz y añadiendo el paquete `script` al camino de imports. Las rutas se leen desde la configuración (regla D5 / D2), así no importa desde qué carpeta se abra el cuaderno.

In [2]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROYECTO = Path.cwd()
while not any((PROYECTO / c).exists() for c in ['db', 'data']) and PROYECTO != PROYECTO.parent:
    PROYECTO = PROYECTO.parent

sys.path.insert(0, str(PROYECTO / 'script'))
from proyecto import config
from proyecto import (cargar, limpiar, diagnostico, transformaciones, cruces,
                      indicadores, validacion, graficos)

def presentar(fig):
    display(fig)

print('raiz del proyecto:', PROYECTO.name)
print('fuentes en data/crudo:', [p.name for p in config.DATA_CRUDO.glob('*.csv')])

raiz del proyecto: mineria
fuentes en data/crudo: ['Average_UK_houseprices_and_salary.csv', 'Income_by_age_and_gender.csv', 'MET_Office_Weather_Data.csv', 'UK_Mortgage_Rate.csv']


## 2. Contracto de rutas (qué se va a usar)

Se listan los archivos que intervienen y su tamaño. La fuente F1 ya viene limpia de una fase anterior; las otras se cargan en crudo desde `data/crudo/`.

In [3]:
for nombre, ruta in [('F1 transacciones', config.RUTA_TRANSACCIONES_LIMPIO),
                    ('F2 hipotecas',     config.RUTA_HIPOTECAS),
                    ('F3 clima',         config.RUTA_CLIMA),
                    ('F4 precio+salario', config.RUTA_PRECIOS_SALARIO),
                    ('F4 ingreso por edad', config.RUTA_INGRESO_EDAD)]:
    print(f"{nombre:<20} {ruta.name:<46} {ruta.stat().st_size / 1e6:,.1f} MB")

F1 transacciones     uk_property_price_limpio.parquet               433.9 MB
F2 hipotecas         UK_Mortgage_Rate.csv                           0.9 MB
F3 clima             MET_Office_Weather_Data.csv                    1.5 MB
F4 precio+salario    Average_UK_houseprices_and_salary.csv          0.0 MB
F4 ingreso por edad  Income_by_age_and_gender.csv                   0.0 MB


## 3. Hipotecas: un vistazo a la oferta de 2022

In [4]:
hipotecas = cargar.cargar_hipotecas()
display(hipotecas.shape)
display(hipotecas.head(3))
print('anios de escaneo:', hipotecas['fecha_escaneo'].dt.year.unique())
print('tipos de producto :', hipotecas['tipo'].unique().tolist())

(7796, 13)

,sku,banco,subtitulo,tipo_crudo,tipo,plazo_anios,tasa_inicial_pct,apr_pct,tasa_reversion,comisiones_total,meses_inicial,fecha_escaneo,tid
index,,,,,,,,,,,,,
0,3739342,Foundation Home Loans,Remortgage,5 year fixed,fixed,25,7.69,7.7,7.24,630.0,60,2022-10-16 06:42:26,483184
1,3738960,Kensington Mortgages,Remortgage,2 year fixed,fixed,25,8.34,8.0,7.35,2347.0,24,2022-10-16 06:42:26,483185
2,3739028,Kensington Mortgages,Remortgage,2 year fixed,fixed,25,8.49,7.9,7.35,108.0,24,2022-10-16 06:42:26,483186


anios de escaneo: [2022]
tipos de producto : ['fixed', 'discounted', 'variable', 'tracker']


## 4. Clima: 36 estaciones, un siglo de lecturas

In [5]:
clima = cargar.cargar_clima()
display(clima.shape)
display(clima.head(3))
print('anios:', clima['anio'].min(), '-', clima['anio'].max(), '| estaciones:', clima['estacion'].nunique())

(37049, 8)

,anio,mes,tmax,tmin,af,lluvia,sol,estacion
0,1941.0,1.0,NaN,NaN,NaN,74.7,NaN,aberporth
1,1941.0,2.0,NaN,NaN,NaN,69.1,NaN,aberporth
2,1941.0,3.0,NaN,NaN,NaN,76.2,NaN,aberporth


anios: 1853.0 - 2020.0 | estaciones: 36


## 5. Precios, salario y brecha por edad

In [6]:
precios = cargar.cargar_precios_salario()
ingreso = cargar.cargar_ingreso_edad()
display(precios.head(3))
print('anios F4a:', precios['anio'].min(), '-', precios['anio'].max())
display(ingreso)
print('grupos de edad:', ingreso['grupo_edad'].unique().tolist())

,anio,precio_promedio_real,salario_mediana_real
0,1975,94983,NaN
1,1976,89281,NaN
2,1977,85028,NaN


anios F4a: 1975 - 2020


,grupo_edad,salario_mediana,genero
0,18 to 21,18392,Male
1,22 to 29,26856,Male
2,30 to 39,34210,Male
3,40 to 49,38463,Male
4,50 to 59,36000,Male
5,60 and over,30944,Male
6,18 to 21,17005,Female
7,22 to 29,25115,Female
8,30 to 39,30540,Female
9,40 to 49,31679,Female


grupos de edad: ['18 to 21', '22 to 29', '30 to 39', '40 to 49', '50 to 59', '60 and over']


## 6. La fuente F1, ya lista en parquet

In [7]:
transacciones = cargar.cargar_precios_limpios()
print('transacciones:', f"{len(transacciones):,}", '| columnas:', transacciones.shape[1])
print('periodo:', transacciones['year'].min(), '-', transacciones['year'].max())
print('regiones:', transacciones['region'].nunique(), '| ciudades:', transacciones['town_city'].nunique())

transacciones: 11,369,413 | columnas: 22
periodo: 2015 - 2026
regiones: 10 | ciudades: 1153


## En resumen

- F2 es un **corte de 2022**: un solo año de escaneo y un plazo fijo de 25 años.
- F3 termina en **2020** y el salario de F4 arranca en **1999**. Encontrar años comunes con F1 deja la ventana de cruce en **2015-2020**.
- La columna de 2022 no se mezcla con la ventana: se analiza aparte como el corte hipotecario.